In [12]:
import os
import tempfile
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Langchain Core Imports

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import(
    RunnablePassthrough,
)

from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader, DirectoryLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

## Load env
load_dotenv()

True

## Data Ingestion And Processing

In [6]:
temp_dir = tempfile.mkdtemp()

pdf_docs = os.listdir("/home/jaywardhan/RAG_Udemy/pdfs")

for pdf in pdf_docs:
    with open(f"/home/jaywardhan/RAG_Udemy/pdfs/{pdf}","rb") as f:
        content = f.read()

    with open(f"{temp_dir}/{pdf}","wb") as file:
        file.write(content)

In [13]:
loader = DirectoryLoader(
    temp_dir,
    glob = "*.pdf",
    loader_cls = PyPDFLoader
)

docs = loader.load()

print(f"Total Docs: {len(docs)}")

for i , doc in enumerate(docs[0:5]):
    print(f"Doc: {i+1}")
    print(f"Context: {doc.page_content}")
    print(f"Metadata: {doc.metadata}\n")


Total Docs: 404
Doc: 1
Context: Web Development for Data Scientists
Why Learn Web Development as a Data Scientist?
Most data science work involves exploring datasets, building models, and
analyzing results. However, in many real-world situations, the value of your work
increases significantly when others can interact with it. Web development provides
a way to make your models and insights accessible through simple user interfaces
or APIs.
Even a basic understanding of web technologies allows you to:
Present your results beyond notebooks
Create simple forms to collect user input
Serve your machine learning models via a web application
Share interactive visualizations or summaries
How Much Web Development is Enough?
As a data scientist, you do not need to become a full-stack web developer. Instead,
it’s more practical to focus on a few key skills that complement your existing
workflow:
HTML to create and structure web pages
CSS to control the appearance and layout
Flask (Python Framework

In [14]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
    separators = [" "],
    length_function = len
)

chunks = text_splitter.split_documents(docs)
print(f"Total Chunks: {len(chunks)}")

Total Chunks: 878


In [15]:
## Setting Up Environment Variables:

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [17]:
## Initializing the embedding model:

embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    dimensions = 1536
)


In [18]:
## Initializing FAISS Vector Store

vectorstore = FAISS.from_documents(
    documents = chunks,
    embedding = embeddings
)

print(f"Total number of vectors stored in faiss vector store are: {vectorstore.index.ntotal}")

Total number of vectors stored in faiss vector store are: 878


In [ ]:
## Saving vectorstore for later use:

vectorstore.save_local("faiss_index") ## Stores the vectorstore in faiss_index directory

In [20]:
## Load Vector Store

loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization = True 
)

print(f"Total numbers of vectors loaded are: {loaded_vectorstore.index.ntotal} vectors")

Total numbers of vectors loaded are: 878 vectors


In [23]:
## Performing Similarity Search:

query = "What is git stash?"
result = vectorstore.similarity_search_with_score(query,k = 3)
result

[(Document(id='89a57f21-f515-433f-bdfc-0e2abaa6b3d6', metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-08-20T15:19:46+00:00', 'moddate': '2025-08-20T15:20:31+00:00', 'source': '/tmp/tmpf3q9igmz/Git&Github.pdf', 'total_pages': 33, 'page': 29, 'page_label': '30'}, page_content='Git Stash - Essential Commands\nWhat is Git Stash?\nGit stash temporarily saves your uncommitted changes so you can work on\nsomething else, then come back and re-apply them later.\nCore Stash Commands\nSave Changes to Stash\nView Stashes\nOutput example:\n# Stash all changes\ngit stash\n# Stash with a message\ngit stash save "work in progress on feature X"\n# Include untracked files\ngit stash -u\n# List all stashes\ngit stash list\nstash@{0}: On main: work in progress on feature X\nstash@{1}: WIP on develop:'),
  np.float32(0.5556855)),
 (Document(id='1a445be3-d9cb-4b9e-b1f0-096b1199c51c', metadata={'producer': 

In [32]:
## Search with metadata filtering

filtered_dict = {"page": 29} ## filtering similarity search results with respect to page number
filtered_results = vectorstore.similarity_search_with_score(
    query,
    k = 3,
    filter = filtered_dict
)

print(f"Filtered Results:\n")
for i , (doc , score) in enumerate(filtered_results):
    print(f"Score: {score:.3f}")
    print(f"Result: {i+1}")
    print(f"Context: {doc.page_content[:200]}...")
    print(f"PageNo: {doc.metadata.get("page")}\n")

Filtered Results:

Score: 0.556
Result: 1
Context: Git Stash - Essential Commands
What is Git Stash?
Git stash temporarily saves your uncommitted changes so you can work on
something else, then come back and re-apply them later.
Core Stash Commands
Sa...
PageNo: 29

Score: 0.641
Result: 2
Context: stashes
git stash list
stash@{0}: On main: work in progress on feature X
stash@{1}: WIP on develop: 5002d47 fix conflict
CodeWithHarry...
PageNo: 29



In [33]:
print(filtered_results)

[(Document(id='89a57f21-f515-433f-bdfc-0e2abaa6b3d6', metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-08-20T15:19:46+00:00', 'moddate': '2025-08-20T15:20:31+00:00', 'source': '/tmp/tmpf3q9igmz/Git&Github.pdf', 'total_pages': 33, 'page': 29, 'page_label': '30'}, page_content='Git Stash - Essential Commands\nWhat is Git Stash?\nGit stash temporarily saves your uncommitted changes so you can work on\nsomething else, then come back and re-apply them later.\nCore Stash Commands\nSave Changes to Stash\nView Stashes\nOutput example:\n# Stash all changes\ngit stash\n# Stash with a message\ngit stash save "work in progress on feature X"\n# Include untracked files\ngit stash -u\n# List all stashes\ngit stash list\nstash@{0}: On main: work in progress on feature X\nstash@{1}: WIP on develop:'), np.float32(0.5556219)), (Document(id='1a445be3-d9cb-4b9e-b1f0-096b1199c51c', metadata={'producer': 'pd

## Build RAG With LCEL

In [54]:
## Using Groq LLM:

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = init_chat_model(model = "groq:openai/gpt-oss-120b")


llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7c76ae258f50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7c76ae259450>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [43]:
## prompt:

simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:

Context: {context}

Question: {question}

Answer: """)

In [44]:
## converting vectorstore into retriever

retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k" : 3}
)

In [45]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7c76bbe8f8c0>, search_kwargs={'k': 3})

In [46]:
from typing import List
def format(docs: List[Document]) -> str:

    formatted = []

    for i, doc in enumerate(docs):
        source = doc.metadata.get('source','unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")

    return "\n\n".join(formatted)

In [55]:
simple_rag_chain = (
    {
        "context": retriever | format,
        "question": RunnablePassthrough() 
    }
    |simple_prompt
    |llm
    |StrOutputParser()
)

In [56]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7c76bbe8f8c0>, search_kwargs={'k': 3})
           | RunnableLambda(format),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer: '), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inp

In [57]:
response = simple_rag_chain.invoke("What are the different ways of resolving git merge conflict?")
response

'**Ways to resolve a Git merge conflict (as described in the provided material)**  \n\n1. **Manual edit of the conflicted file**  \n   - Open the file that contains the conflict markers (`<<<<<<<`, `=======`, `>>>>>>>`).  \n   - Decide which changes (or a combination of both) you want to keep.  \n   - Delete the conflict‑marker lines and any unwanted code.  \n   - Stage the file (`git add <file>`) and commit the resolution.\n\n2. **Using the “ours” vs. “theirs” shortcuts**  \n   - **Keep the current‑branch version (“ours”)** – discards the incoming changes:  \n     ```bash\n     git checkout --ours <file>      # replaces the file with the version from the branch you are on\n     ```\n   - **Keep the incoming‑branch version (“theirs”)** – discards the current‑branch changes:  \n     ```bash\n     git checkout --theirs <file>    # replaces the file with the version from the branch being merged\n     ```\n   - After choosing, stage the file and commit.\n\n3. **Using a merge‑tool**  \n   -

In [59]:
## Streaming RAG Chain:

streaming_rag_chain = (
    {
        "context": retriever | format,
        "question": RunnablePassthrough() 
    }
    |simple_prompt
    |llm
)

stream_response = streaming_rag_chain.invoke("What are types of Machine Learning?")
stream_response

AIMessage(content='Based on the provided documents, Machine\u202fLearning can be broken down into a few distinct “types” or sub‑categories:\n\n1. **Traditional Machine Learning** – the general approach of giving computers data so they can learn patterns and make predictions (e.g., using models such as Random Forests or Support Vector Machines).\n\n2. **Deep Learning** – a specific type of Machine Learning that relies on **neural‑network** models that learn in multiple layers, mimicking the way the brain works.\n\n3. **Neural‑Network‑based Machine Learning** – sometimes described as its own type because it uses neural‑network architectures even outside the deep‑learning context.\n\nSo, the main types of Machine Learning mentioned in the context are **traditional ML** (with models like Random Forests and SVMs) and **Deep Learning**, which is the neural‑network‑based variant of ML.', additional_kwargs={'reasoning_content': 'We need to answer based only on the given context. The question: 

In [61]:
streaming_rag_chain2 = (
    {
        "context": retriever | format,
        "question": RunnablePassthrough()
    }
    | simple_prompt
    | llm
    | StrOutputParser()
)

for chunk in streaming_rag_chain2.stream(
    "What are types of Machine Learning?"
):
    print(chunk, end="")

Based on the provided documents, the machine‑learning approaches that are highlighted fall into a few distinct categories (often referred to as “types” of ML in this material):

1. **Neural Networks** – the core of deep learning, which is described as a tool that uses layered networks that learn like the brain.  
2. **Random Forests** – an ensemble‑based model that builds many decision trees and aggregates their predictions.  
3. **Support Vector Machines (SVMs)** – a model that finds optimal separating hyper‑planes for classification or regression tasks.  

These three model families are the primary “types” of machine‑learning methods mentioned in the context.